### Candidate explanations

- A cluster around ±0.10 could reflect source precision or a posting convention.
- Larger differences could reflect ordering, sign assumptions, or incomplete
  history. A missing transaction is one hypothesis, not an established finding.

Use the following diagnostics to distinguish these explanations.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys 
sys.path.append("..")

from src.spark_utils import load_config, get_spark, read_from_mysql
from pyspark.sql import functions as F
from pyspark.sql import Window

cfg = load_config()
spark = get_spark(cfg["jdbc_jar"])

fact_transactions = read_from_mysql(
    spark, cfg["jdbc_url"], cfg["db_user"], cfg["db_password"], "fact_transactions"
)

In [ ]:
from pyspark.sql.functions import when


w_naive = Window.partitionBy("account_id").orderBy("date_key", "trans_id")

with_running = (fact_transactions
    .withColumn("signed_amount",
                F.when(F.col("type") == "PRIJEM", F.col("amount")).otherwise(-F.col("amount")))
    .withColumn("computed_balance",
                F.sum("signed_amount").over(w_naive))            
)

balance_mismatches = with_running.filter(
    F.abs(F.col("computed_balance") - F.col("balance")) > F.lit(0.01)
)

print("Mismatched rows:", balance_mismatches.count())

## Step 1 — is the failure cascading from a single point per account?

In [ ]:
first_fail_per_account = (balance_mismatches
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("account_id").orderBy("date_key", "trans_id")
    ))
    .filter(F.col("rn") == 1))

print("Accounts with at least one difference:", first_fail_per_account.count())
first_fail_per_account.select(
    "account_id", "date_key", "trans_id", "type", "amount", "balance", "computed_balance"
).show(10)

### Candidate explanations

- A cluster around ±0.10 could reflect source precision or a posting convention.
- Larger differences could reflect ordering, sign assumptions, or incomplete
  history. A missing transaction is one hypothesis, not an established finding.

Use the following diagnostics to distinguish these explanations.


## Step 2 — pairwise check: does the mismatch depend on cumulative history?

The following exploratory calculation rounds to whole units because `F.round`
has no scale argument. It cannot diagnose 0.10-scale differences; use the later
two-decimal pairwise calculation for that comparison.


In [ ]:
w = Window.partitionBy("account_id").orderBy("date_key", "trans_id")

pairwise_check = (fact_transactions
    .withColumn("signed_amount",
                F.when(F.col("type") == "PRIJEM", F.col("amount")).otherwise(-F.col("amount")))
    .withColumn("prev_balance", F.lag("balance").over(w))
    .withColumn("expected_balance", F.col("prev_balance") + F.col("signed_amount"))
    .withColumn("pairwise_diff", F.round(F.col("balance") - F.col("expected_balance")))
)

pairwise_check.filter(F.col("pairwise_diff") != 0).groupBy("pairwise_diff").count().orderBy(F.desc("count")).show(20)

## Step 3 — are same-day multi-transaction combinations actually common?

In [ ]:
smae_day_multi = (
    fact_transactions
    .groupBy("account_id", "date_key")
    .count()
    .filter(F.col("count") > 1)
)

print("Account+day combinations with multiple transactions:", smae_day_multi.count())

## Step 4 — check at the day level instead of the transaction level

This diagnostic treats the largest transaction ID as the last posting of the day.
That is an assumption under investigation, so the resulting daily differences
are not independent evidence of the true posting order.


In [ ]:
last_of_day = (
    fact_transactions
    .withColumn("signed_amount", F.when(F.col("type") == "PRIJEM", F.col("amount")).otherwise(-F.col("amount")))
    .withColumn("rn_desc", F.row_number().over(
        Window.partitionBy("account_id", "date_key").orderBy(F.desc("trans_id"))))
)

daily_summary = (last_of_day
    .groupBy("account_id", "date_key")
    .agg(F.sum("signed_amount").alias("daily_net_change")))

daily_end_balance = (last_of_day
    .filter(F.col("rn_desc") == 1)
    .select("account_id", "date_key", F.col("balance").alias("last_balance_of_day")))

daily_check = (daily_summary
    .join(daily_end_balance, ["account_id", "date_key"])
    .withColumn("prev_day_balance", F.lag("last_balance_of_day").over(
        Window.partitionBy("account_id").orderBy("date_key")))
    .withColumn("day_diff", F.round(
        F.col("last_balance_of_day") - (F.col("prev_day_balance") + F.col("daily_net_change")), 2))
)

daily_check.filter(F.col("day_diff") != 0).count()    

In [ ]:
single_txn_days = (fact_transactions
    .groupBy("account_id", "date_key")
    .count()
    .filter(F.col("count") == 1)
    .select("account_id", "date_key"))

single_day_failures = (daily_check
    .join(single_txn_days, ["account_id", "date_key"])
    .filter(F.col("day_diff") != 0))

print("Single-transaction days with a mismatch:", single_day_failures.count())

## Step 5 — test the "sign flip" hypothesis on single-transaction days

In [ ]:
sign_flip_check = (single_day_failures
    .withColumn("is_sign_flip", F.col("day_diff") == -2 * F.col("daily_net_change")))

sign_flip_check.groupBy("is_sign_flip").count().show()

(sign_flip_check
    .filter(F.col("is_sign_flip"))
    .join(fact_transactions, ["account_id", "date_key"])
    .groupBy("type")
    .count()
    .show())

## Step 6 — Historical manual reconstruction in DBeaver

Earlier notes describe account `1033` on 1998-06-30:

- Deposit: transaction `303570`, `PRIJEM` / `VKLAD`, amount 6,463.
- Interest: transaction `3440301`, `PRIJEM` / `UROK`, amount 163.8.
- With interest first: 30,432.2 + 163.8 = 30,596.0, compared with a stated
  interest-row balance of 30,595.9. The 0.10 difference **still exceeds** the
  validation tolerance of 0.01.
- Starting from that stated balance: 30,595.9 + 6,463 = 37,058.9, matching
  the deposit-row balance reported in the notes.

This sample supports investigating interest-first ordering. Different ID ranges
do not prove a separate monthly batch process or establish the correct order for
every account. The original SQL output is not retained here; reproduce this
example before citing it as independently verified evidence.


## Step 7 — Compare the implemented ordering and inspect residual differences

Run the revised check with the unchanged 0.01 tolerance. Compare its mismatch
count with the naive baseline, then inspect cumulative and pairwise differences.
Keep remaining failures visible; do not interpret an expected failure as a pass.


In [ ]:
from src.validations import check_running_balance

balance_result_v2, balance_mismatches_v2 = check_running_balance(fact_transactions)
print(balance_result_v2)

In [ ]:
remaining = balance_mismatches_v2.withColumn(
    "diff", F.round(F.col("computed_balance") - F.col("balance"), 2)
)
remaining.groupBy("diff").count().orderBy(F.desc("count")).show(10)
remaining.groupBy((F.col("k_symbol") == "UROK")).count().show()

In [ ]:
w_fixed = Window.partitionBy("account_id").orderBy(
    "date_key",
    F.when(F.col("k_symbol") == "UROK", F.lit(0)).otherwise(F.lit(1)),
    "trans_id"
)

pairwise_fixed = (fact_transactions
    .withColumn("signed_amount", F.when(F.col("type") == "PRIJEM", F.col("amount")).otherwise(-F.col("amount")))
    .withColumn("prev_balance", F.lag("balance").over(w_fixed))
    .withColumn("expected_balance", F.col("prev_balance") + F.col("signed_amount"))
    .withColumn("pairwise_diff", F.round(F.col("balance") - F.col("expected_balance"), 2))
)

pairwise_fixed.filter(F.col("pairwise_diff") != 0).groupBy(F.col("k_symbol") == "UROK").count().show()